In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path('~/Documents/metrology_ir/TRELLIS.2').expanduser()))


import os
os.environ['OPENCV_IO_ENABLE_OPENEXR'] = '1'
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # Can save GPU memory
import cv2
import imageio
from PIL import Image
import torch
from trellis2.pipelines import Trellis2ImageTo3DPipeline
from trellis2.utils import render_utils
from trellis2.renderers import EnvMap
import o_voxel

/home/ahc/miniconda3/envs/trellis2/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[SPARSE] Conv backend: flex_gemm; Attention backend: flash_attn


In [2]:

# 1. Setup Environment Map
envmap = EnvMap(torch.tensor(
    cv2.cvtColor(cv2.imread('TRELLIS.2/assets/hdri/forest.exr', cv2.IMREAD_UNCHANGED), cv2.COLOR_BGR2RGB),
    dtype=torch.float32, device='cuda'
))

# 2. Load Pipeline
pipeline = Trellis2ImageTo3DPipeline.from_pretrained("microsoft/TRELLIS.2-4B")
pipeline.cuda()

[ATTENTION] Using backend: flash_attn


/home/ahc/miniconda3/envs/trellis2/lib/python3.10/site-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/home/ahc/miniconda3/envs/trellis2/lib/python3.10/site-packages/timm/models/registry.py:4: FutureWarning: Importing from timm.models.registry is deprecated, please import via timm.models
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.models", FutureWarning)


In [3]:
VIZ_MASKED = Path("viz_masked")
OUT_DIR = Path("trellis2_outputs")
OUT_DIR.mkdir(exist_ok=True)

for img_path in sorted(VIZ_MASKED.glob("*_masked.png")):
    stem = img_path.stem.replace("_masked", "")
    print(f"\n=== {stem} ===")

    image = Image.open(img_path)
    mesh = pipeline.run(image)[0]
    mesh.simplify(16777216)

    video = render_utils.make_pbr_vis_frames(render_utils.render_video(mesh, envmap=envmap))
    imageio.mimsave(str(OUT_DIR / f"{stem}.gif"), video, fps=15)
    print(f"  saved {stem}.gif")

    glb = o_voxel.postprocess.to_glb(
        vertices            =   mesh.vertices,
        faces               =   mesh.faces,
        attr_volume         =   mesh.attrs,
        coords              =   mesh.coords,
        attr_layout         =   mesh.layout,
        voxel_size          =   mesh.voxel_size,
        aabb                =   [[-0.5, -0.5, -0.5], [0.5, 0.5, 0.5]],
        decimation_target   =   1000000,
        texture_size        =   4096,
        remesh              =   True,
        remesh_band         =   1,
        remesh_project      =   0,
        verbose             =   False,
    )
    glb.export(str(OUT_DIR / f"{stem}.glb"), extension_webp=True)
    print(f"  saved {stem}.glb")



=== blocks_scene006 ===


Rendering:   0%|          | 0/120 [00:00<?, ?it/s]/home/ahc/miniconda3/envs/trellis2/lib/python3.10/site-packages/utils3d/torch/nerf.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=False):
/home/ahc/miniconda3/envs/trellis2/lib/python3.10/site-packages/torch/nn/functional.py:5015: UserWarning: Default grid_sample and affine_grid behavior has changed to align_corners=False since 1.3.0. Please specify align_corners=True if the old behavior is desired. See the documentation of grid_sample for details.
  warnings.warn(
Rendering: 100%|██████████| 120/120 [00:39<00:00,  3.02it/s]


  saved blocks_scene006.gif


/home/ahc/miniconda3/envs/trellis2/lib/python3.10/site-packages/cumesh/remeshing.py:220: UserWarning: Using torch.cross without specifying the dim arg is deprecated.
Please either pass the dim explicitly or simply use torch.linalg.cross.
The default value of dim will change to agree with that of linalg.cross in a future release. (Triggered internally at /pytorch/aten/src/ATen/native/Cross.cpp:62.)
  normals0 = torch.cross(mesh_vertices[atempt_triangles_0[:, 1]] - mesh_vertices[atempt_triangles_0[:, 0]], mesh_vertices[atempt_triangles_0[:, 2]] - mesh_vertices[atempt_triangles_0[:, 0]])


  saved blocks_scene006.glb

=== cactus_scene007 ===


...

  saved grogu_scene002.gif
  saved grogu_scene002.glb

=== pitcher_scene005 ===


Rendering: 100%|██████████| 120/120 [00:49<00:00,  2.43it/s]


  saved pitcher_scene005.gif
  saved pitcher_scene005.glb

=== teapot_scene001 ===


Rendering: 100%|██████████| 120/120 [00:39<00:00,  3.04it/s]


  saved teapot_scene001.gif
  saved teapot_scene001.glb
